# Advanced AI Lab 1 — Sentiment Analysis

This is the **single entry point** for all Lab 1 experiments.  
Run the cells top-to-bottom in each section you need.  
All heavy logic lives in the project files — this notebook just calls them.

| Section | Task |
|---|---|
| 0 — Setup | Check GPU, install dependencies, wandb login |
| 1 — Task 1.1 ANN | Simple ANN on small (1 K) and large (25 K) datasets |
| 2 — Task 1.1 BiLSTM | Bidirectional LSTM on small and large datasets |
| 3 — Task 1.2 BERT | Fine-tune BERT on the large Amazon dataset |
| 4 — Task 1.2 DistilBERT | Fine-tune DistilBERT on the large Amazon dataset |
| 5 — Grade 5 | BERT + DistilBERT on the public ~1 GB amazon_polarity dataset |
| 6 — Comparison | Side-by-side results across all models |
| 7 — W&B | View training curves at https://wandb.ai |

---
## Section 0 — Setup

Run this cell once before anything else.  
It verifies PyTorch, shows the GPU (if available), and confirms all modules load correctly.

In [1]:
import sys
import os

# Make sure the Lab 1 root is on the Python path so all imports resolve
project_root = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, project_root)

import torch
import config

print("=" * 55)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Device in use   : {config.DEVICE}")
print("=" * 55)
print()
print("Datasets found:")
print(f"  Small (1 K)  : {os.path.exists(config.SMALL_DATASET_PATH)}  →  {config.SMALL_DATASET_PATH}")
print(f"  Large (25 K) : {os.path.exists(config.LARGE_DATASET_PATH)}  →  {config.LARGE_DATASET_PATH}")
print()
print("All project modules are ready.")
print("Run any section below to start an experiment.")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2080 Ti
Device in use   : cuda

Datasets found:
  Small (1 K)  : True  →  /Labs/Lab1/amazon_cells_labelled.txt
  Large (25 K) : True  →  /Labs/Lab1/amazon_cells_labelled_LARGE_25K.txt

All project modules are ready.
Run any section below to start an experiment.


### Section 0.1 — Weights & Biases Login (run once)

All experiments log to **https://wandb.ai** automatically.  
Run the cell below to install wandb (if missing) and authenticate.  
Get your API key at **https://wandb.ai/authorize**

In [2]:
import subprocess, sys

import wandb
wandb.login()   # prompts for API key if not already stored

print()
print("Logged in to wandb.")
print("After running experiments, view all results at: https://wandb.ai")
print(f"Project name: {__import__('config').WANDB_PROJECT}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: deeplab15group (deeplab15group-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Logged in to wandb.
After running experiments, view all results at: https://wandb.ai
Project name: advanced-ai-lab-1


---
## Section 1 — Task 1.1: Simple ANN

A feed-forward network trained on **TF-IDF bag-of-words features**.  
We run the same architecture twice — once on the 1 K small dataset, once on the 25 K large dataset — to observe how training-data volume impacts accuracy.

| Cell | Dataset | What we learn |
|---|---|---|
| 1.1 | Small (1 K) | Baseline — limited data |
| 1.2 | Large (25 K) | Effect of 25× more training data |

In [3]:
# 1.1  Simple ANN — small dataset (1 K reviews)
from experiments.task01_ann import main as run_ann

ann_small = run_ann(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [small dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 4,033 features
  Split: 700 train / 150 val / 150 test

Building ANN  (vocab_size=4,033) …
  [build_ann] dataset='small' → SmallANN (2 blocks, 128→64)
  Total parameters    : 524,994
  Trainable parameters: 524,994



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Small
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0005
  Epochs      : 50
  Patience    : 3
  Trainable   : 524,994 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/50  |  train  loss=0.7231  acc=47.7%  |  val  loss=0.6929  acc=50.0%  f1=0.6667  prec=0.5000  rec=1.0000


  Epoch  2/50  |  train  loss=0.6846  acc=54.6%  |  val  loss=0.6792  acc=55.3%  f1=0.6825  prec=0.5294  rec=0.9600


  Epoch  3/50  |  train  loss=0.6499  acc=62.1%  |  val  loss=0.6494  acc=64.7%  f1=0.6788  prec=0.6222  rec=0.7467


...

  Epoch 13/50  |  train  loss=0.0387  acc=99.0%  |  val  loss=0.5166  acc=79.3%  f1=0.7974  prec=0.7821  rec=0.8133


  Epoch 14/50  |  train  loss=0.0374  acc=98.7%  |  val  loss=0.5139  acc=80.7%  f1=0.8079  prec=0.8026  rec=0.8133
  Early stopping triggered (no improvement for 3 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 75.33%   F1 : 0.7643   Precision : 0.7317   Recall : 0.8000
──────────────────────────────────────────────────────────────



batch_loss,█▇▇▆▇▆▇▆▇▆▆▆▆▄▃▃▃▂▂▃▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▂▁▁▁
epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
train_accuracy,▁▂▃▅▇▇████████
train_loss,██▇▆▅▃▂▁▁▁▁▁▁▁
val_accuracy,▁▂▄▆▇▇▇███████
val_f1,▁▂▂▅▆▆▇▇▇███▇█
val_loss,██▇▅▂▁▁▁▂▃▂▃▂▂
val_precision,▁▂▄▆▇▇▇▇█▇████
val_recall,█▇▁▂▁▁▃▃▂▄▃▃▃▃
batch_loss,0.04712
epoch,14


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Small.pth
[Result] Experiment       : Task01_ANN_Small
[Result] Best Val Accuracy: 80.67%
[Result] Test Accuracy    : 75.33%
[Result] Test F1          : 0.7643



In [4]:
# 1.2  Simple ANN — large dataset (25 K reviews)
from experiments.task01_ann import main as run_ann

ann_large = run_ann(dataset="large")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [large dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 10,000 features
  Split: 17,510 train / 3,740 val / 3,750 test

Building ANN  (vocab_size=10,000) …
  [build_ann] dataset='large' → LargeANN (3 blocks, 512→256→64)
  Total parameters    : 5,269,954
  Trainable parameters: 5,269,954



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Large
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=5e-05
  Epochs      : 20
  Patience    : 2
  Trainable   : 5,269,954 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/20  |  train  loss=0.6664  acc=58.5%  |  val  loss=0.5636  acc=77.6%  f1=0.8390  prec=0.7429  rec=0.9637


  Epoch  2/20  |  train  loss=0.4932  acc=77.6%  |  val  loss=0.3770  acc=86.0%  f1=0.8839  prec=0.8855  rec=0.8824


  Epoch  3/20  |  train  loss=0.3211  acc=87.5%  |  val  loss=0.3200  acc=86.3%  f1=0.8826  prec=0.9126  rec=0.8545


...

  Epoch  5/20  |  train  loss=0.1452  acc=95.1%  |  val  loss=0.3397  acc=86.5%  f1=0.8860  prec=0.9065  rec=0.8664


  Epoch  6/20  |  train  loss=0.0998  acc=96.8%  |  val  loss=0.3744  acc=85.8%  f1=0.8793  prec=0.9030  rec=0.8567
  Early stopping triggered (no improvement for 2 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 86.59%   F1 : 0.8862   Precision : 0.9099   Recall : 0.8637
──────────────────────────────────────────────────────────────



batch_loss,█▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▅▆▄▄▂▃▂▃▂▂▂▃▂▃▂▁▂▂▂▁▂▁▂▁
epoch,▁▂▄▅▇█
train_accuracy,▁▄▆▇██
train_loss,█▆▄▂▂▁
val_accuracy,▁▇███▇
val_f1,▁█▇██▇
val_loss,█▃▁▁▂▃
val_precision,▁▇████
val_recall,█▃▁▂▂▁
batch_loss,0.07763
epoch,6


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Large.pth
[Result] Experiment       : Task01_ANN_Large
[Result] Best Val Accuracy: 86.63%
[Result] Test Accuracy    : 86.59%
[Result] Test F1          : 0.8862



In [5]:
# grade 5  Simple ANN — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_ann import main as run_ann

ann_public = run_ann(dataset="public")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [public dataset] …
  (This may take a while on first run — the dataset is ~1 GB)


  Loaded 3,600,000 examples from 'amazon_polarity'
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 10,000 features
  Split: 2,521,440 train / 538,560 val / 540,000 test

Building ANN  (vocab_size=10,000) …
  [build_ann] dataset='public' → LargeANN (3 blocks, 512→256→64)
  Total parameters    : 5,269,954
  Trainable parameters: 5,269,954



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Public
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0001
  Epochs      : 10
  Patience    : 2
  Trainable   : 5,269,954 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/10  |  train  loss=0.3030  acc=86.5%  |  val  loss=0.2388  acc=90.3%  f1=0.9032  prec=0.8998  rec=0.9065


  Epoch  2/10  |  train  loss=0.2541  acc=89.8%  |  val  loss=0.2324  acc=90.6%  f1=0.9068  prec=0.8949  rec=0.9191


...

  Epoch  9/10  |  train  loss=0.2189  acc=91.5%  |  val  loss=0.2091  acc=91.7%  f1=0.9172  prec=0.9155  rec=0.9189


  Epoch 10/10  |  train  loss=0.2061  acc=92.2%  |  val  loss=0.2068  acc=91.8%  f1=0.9184  prec=0.9150  rec=0.9219



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 91.84%   F1 : 0.9187   Precision : 0.9158   Recall : 0.9216
──────────────────────────────────────────────────────────────



batch_loss,▆▆▇▃▄▄█▇▅▄▃▂▅▆▅▅▆▂▂▆▄▄▅▄▄▃▆▅▇▄▆▅▃█▃▁▁▄▄█
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▅▅▅▆▆▆▇▇█
train_loss,█▄▄▄▄▃▃▃▂▁
val_accuracy,▁▂▃▃▅▅▆▇▇█
val_f1,▁▃▃▂▅▅▆▇▇█
val_loss,█▇▆▆▄▄▃▂▂▁
val_precision,▂▁▅█▅▆▆▇▆▆
val_recall,▅▇▅▁▆▅▆▆▇█
batch_loss,0.23771
epoch,10


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Public.pth
[Result] Experiment       : Task01_ANN_Public
[Result] Best Val Accuracy: 91.81%
[Result] Test Accuracy    : 91.84%
[Result] Test F1          : 0.9187



---
## Section 2 — Task 1.1: Bidirectional LSTM

A BiLSTM with **learned word embeddings** that reads the sentence in both directions.  
Unlike the ANN, the BiLSTM preserves word order and captures long-range dependencies  
(e.g. negations, emphasis) — important signals for sentiment.

| Cell | Dataset | What we learn |
|---|---|---|
| 2.1 | Small (1 K) | Sequence model on limited data |
| 2.2 | Large (25 K) | Sequence model vs ANN on the same 25 K data |

In [ ]:
# 2.1  BiLSTM — small dataset (1 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_small = run_bilstm(dataset="small")

In [ ]:
# 2.2  BiLSTM — large dataset (25 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_large = run_bilstm(dataset="large")

In [ ]:
# grade 5  BiLSTM — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_bilstm import main as run_bilstm

bilstm_public = run_bilstm(dataset="public")

---
## Section 3 — Task 1.2: BERT Fine-Tuning

**BERT-base-uncased** was pre-trained on BookCorpus + English Wikipedia using  
masked language modelling and next-sentence prediction.  
We replace its classification head and fine-tune **all 12 transformer layers** for sentiment.

> ⚠ GPU strongly recommended — BERT has 110 M parameters.  
> On CPU, reduce `"epochs"` in `config.py → BERT_CONFIG` to `1`.

In [ ]:
# 3.1  BERT — fine-tuned on the large Amazon dataset (25 K reviews)
from experiments.task02_bert import main as run_bert

bert_results = run_bert()

---
## Section 4 — Task 1.2: DistilBERT Fine-Tuning

**DistilBERT-base-uncased** is a knowledge-distilled version of BERT:  
40% fewer parameters, 60% faster, ~97% of BERT's accuracy.  

Running both side-by-side provides a direct **complexity vs accuracy trade-off** study.

In [ ]:
# 4.1  DistilBERT — fine-tuned on the large Amazon dataset (25 K reviews)
from experiments.task02_distilbert import main as run_distilbert

distilbert_results = run_distilbert()

---
## Section 5 — Grade 5: Two Transformers on the Public Dataset (~1 GB)

Runs **BERT-base-uncased** and **DistilBERT-base-uncased** back-to-back,  
both fine-tuned on `amazon_polarity` from Hugging Face.

- **amazon_polarity** — ~3.6 M product reviews, ~1 GB download  
- Training is capped at `PUBLIC_MAX_SAMPLES` rows (set in `config.py`, default 100 K)  
- Both models train on the identical data split for a fair comparison

> ⚠ GPU strongly recommended.  On CPU, set `"epochs": 1` in
> `BERT_PUBLIC_CONFIG` and `DISTILBERT_PUBLIC_CONFIG` in `config.py`.

In [ ]:

# 5.1  BERT + DistilBERT on the public ~1 GB amazon_polarity dataset
from experiments.grade5_transformers_public import main as run_grade5

grade5_results = run_grade5()


---
## Section 6 — Task 1.3: Standardized Comparison (Required by Spec)

This section satisfies the Task 1.3 requirement:
> *"Use the **same test dataset** for Simple ANN, LSTM-based model and the Transformer."*

The cell below trains all three models on the **25 K large dataset**, evaluates them on the **exact same 15 % test split**, and prints:

- Test accuracy and F1 score for each model
- Parameter count (model complexity)
- Relative inference speed (compared to the ANN baseline)
- A written discussion of all three Task 1.3 comparison questions

The standardization is guaranteed by `RANDOM_SEED = 42` in `config.py` — every model's data loader applies the same stratified split to the same raw texts.

> ⚠ This cell trains all three models sequentially. Run Sections 1–5 first if you want individual results, or run this cell alone for the complete standardized comparison.

In [ ]:
# Task 1.3 — Standardized comparison: Simple ANN vs BiLSTM vs BERT on the identical test split
from experiments.task03_comparison import main as run_task03_comparison
comparison_results = run_task03_comparison()

---
## Section 6 — Model Comparison

Print a side-by-side table of all experiment results.  
Requires the cells above to have been run in the same session.

In [ ]:
# 6.1  Print side-by-side comparison of all models
results_dict = {}

try: results_dict["ANN  (small 1K)"]    = ann_small
except NameError: pass
try: results_dict["ANN  (large 25K)"]   = ann_large
except NameError: pass
try: results_dict["ANN  (public 100K)"] = ann_public
except NameError: pass
try: results_dict["BiLSTM (small 1K)"]  = bilstm_small
except NameError: pass
try: results_dict["BiLSTM (large 25K)"] = bilstm_large
except NameError: pass
try: results_dict["BiLSTM (public)"]    = bilstm_public
except NameError: pass
try: results_dict["BERT   (large 25K)"] = bert_results
except NameError: pass
try: results_dict["DistilBERT (25K)"]   = distilbert_results
except NameError: pass
try: results_dict["BERT   (public)"]    = grade5_results.get("BERT", {})
except (NameError, AttributeError): pass
try: results_dict["DistilBERT (pub)"]   = grade5_results.get("DistilBERT", {})
except (NameError, AttributeError): pass

print(f"{'Model':<25} {'Test Acc':>10}  {'Test F1':>10}  {'Best Val Acc':>12}")
print("-" * 62)
for name, res in results_dict.items():
    print(
        f"  {name:<23}"
        f"  {res.get('test_accuracy', float('nan')):>8.2f}%"
        f"  {res.get('test_f1', float('nan')):>10.4f}"
        f"  {res.get('best_val_accuracy', float('nan')):>10.2f}%"
    )
print("-" * 62)
print("\nView training curves at https://wandb.ai  (project: advanced-ai-lab-1)")

---
## Section 7 — Weights & Biases: View All Results

All experiments automatically stream metrics to **https://wandb.ai** — no local server needed.

### Viewing results
Open **https://wandb.ai** in any browser — all runs are grouped under project **`advanced-ai-lab-1`**.  
Use the **Charts** panel to compare training curves, val F1, and test accuracy side-by-side.

### Key comparisons to make
- **ANN small vs large** — impact of dataset size on bag-of-words features  
- **ANN vs BiLSTM** — bag-of-words vs sequence model on the same data  
- **BiLSTM vs BERT** — trained-from-scratch vs pre-trained transformer  
- **BERT vs DistilBERT** — accuracy trade-off for model compression  

> To rename the project, change `WANDB_PROJECT` in `config.py`.